In [1]:
## Image viewing
import os
from pathlib import Path

import navis


import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import seaborn as sns

import colorcet as cc

IBMblue = "648FFF"
IBMpurple = "785EF0"
IBMhotpink = "DC267F"
IBMorange = "FE6100"
IBMyellow = "FFB000"

# Force PyImageJ to use Conda's Java, ignoring system defaults.
# Must be set BEFORE scyjava/imagej is imported (those trigger JVM startup).
if "CONDA_PREFIX" in os.environ:
    os.environ["JAVA_HOME"] = os.environ["CONDA_PREFIX"]

# Ensure the JVM is started in headless mode from the very first call.
# In a Jupyter kernel the JVM only starts once; if it comes up without the
# right options, the IJ1 legacy layer ends up 'Inactive' and IJ.openImage()
# (which the H5J_Loader_Plugin relies on) silently returns null.
import scyjava
scyjava.config.add_option("-Djava.awt.headless=true")

import imagej

# FIJI_APP = "/home/william-zheng/Downloads/Fiji.app"
# PROJECT_DIR = "/home/william-zheng/Documents/Programming/Python/NeuroInformatics/summer_2026/neuroinfo_fruitfly"
FIJI_APP = "/Users/vuhepola/Desktop/Fiji"
PROJECT_DIR = Path.cwd()
DATA_DIR = PROJECT_DIR / "data"
FISBe_DIR = DATA_DIR / "FISBe"
FlyLight_DIR = DATA_DIR / "FlyLight"
FANC_DIR = DATA_DIR / "FANC"

print(FIJI_APP)
print(PROJECT_DIR)
print(DATA_DIR)
print(FISBe_DIR)
print(FlyLight_DIR)
print(FANC_DIR)

/Users/vuhepola/Desktop/Fiji
/Users/vuhepola/GitHub/Repos/neuroinfo_fruitfly
/Users/vuhepola/GitHub/Repos/neuroinfo_fruitfly/data
/Users/vuhepola/GitHub/Repos/neuroinfo_fruitfly/data/FISBe
/Users/vuhepola/GitHub/Repos/neuroinfo_fruitfly/data/FlyLight
/Users/vuhepola/GitHub/Repos/neuroinfo_fruitfly/data/FANC


In [2]:
# import zarr
# import sys
# import napari

# # raw = zarr.load(sys.argv[1], path="volumes/raw")
# # gts = zarr.load(sys.argv[1], path="volumes/gt_instances")

# file = "R38F04-20181005_63_G3"

# file_name = f"{file}.zarr"

# raw = zarr.load(FISBe_DIR/ "completely/train" / f"{file_name}/volumes/raw")
# gts = zarr.load(FISBe_DIR/ "completely/train" / f"{file_name}/volumes/gt_instances")

# viewer = napari.Viewer(ndisplay=3)

# for idx, gt in enumerate(gts):
#     viewer.add_labels(
#         gt,
#         rendering='translucent', 
#         blending='additive', 
#         name=f'gt_{idx}',
#     )

# viewer.add_image(
#     raw[0], 
#     colormap="red", 
#     name="MCFO Red",
#     blending='additive',
# )
# viewer.add_image(
#     raw[1], 
#     colormap="green", 
#     name="MCFO Green",
#     blending='additive',
# )
# viewer.add_image(
#     raw[2], 
#     colormap="blue", 
#     name="MCFO Blue",
#     blending='additive',
# )

# napari.run()

In [25]:
import zarr
import dask.array as da
import sys
import napari
from pathlib import Path

# Assuming FISBe_DIR is defined somewhere in your environment
# FISBe_DIR = Path(...)

file = "R38F04-20181005_63_G3"
file_name = f"{file}.zarr"
base_path = FISBe_DIR / "completely/train" / file_name

# 1. USE LAZY LOADING
# Instead of zarr.load(), use zarr.open() wrapped in a Dask array.
# This tells napari to only load the chunks of data currently visible on screen.
raw_zarr = zarr.open(str(base_path / "volumes/raw"), mode='r')
raw_dask = da.from_zarr(raw_zarr)

gts_zarr = zarr.open(str(base_path / "volumes/gt_instances"), mode='r')
gts_dask = da.from_zarr(gts_zarr)

viewer = napari.Viewer(ndisplay=3)

# 2. OPTIMIZE IMAGE RENDERING
# Using 'mip' (Maximum Intensity Projection) is significantly faster for 3D 
# volumes than the default 'mip' or 'translucent' algorithms.
colors = ["red", "green", "blue"]

# Find the middle Z-slice index to use as a representative sample
mid_z = raw_dask.shape[1] // 2

for i in range(3):
    # 1. Grab a single 2D slice from the center of the volume and load it into RAM
    sample_slice = raw_dask[i, mid_z, :, :].compute()
    
    # 2. Compute robust limits ignoring the lowest 1% (background) and highest 0.1% (noise/hot pixels)
    # Adjust 99.9 to 99.0 if the image is still slightly too bright
    p_low = np.percentile(sample_slice, 1)      
    p_high = np.percentile(sample_slice, 99.9)  
    
    # Fallback to prevent an error if the channel or slice is completely empty
    if p_low == p_high:
        p_high = p_low + 1

    # 3. Pass the custom contrast limits to napari
    viewer.add_image(
        raw_dask[i], 
        colormap=colors[i], 
        name=f"MCFO {colors[i].capitalize()}",
        blending='additive',
        rendering='mip',
        contrast_limits=(p_low, p_high) # <-- This forces napari to scale the brightness correctly
    )

# 3. LABEL LAYERS WARNING
# Rendering many individual translucent label layers in 3D is extremely GPU-heavy.
# If it is still slow, consider combining your ground truth instances into a single 
# array where each instance has a unique integer ID, rather than a list of separate arrays.
for idx, gt in enumerate(gts_dask):
    viewer.add_labels(
        gt,
        rendering='translucent', # Note: 'translucent' in 3D is slow. Try 'iso_categorical' if available.
        blending='additive', 
        name=f'gt_{idx}',
    )

napari.run()

## SWC Files (Gaurav)


In [18]:
output_path = FISBe_DIR /"completely/train"/ "swc_output"
print(output_path)

/Users/vuhepola/GitHub/Repos/neuroinfo_fruitfly/data/FISBe/completely/train/swc_output


In [19]:
nl = navis.read_swc(output_path / f"{file}/*.swc")

Importing:   0%|          | 0/2 [00:00<?, ?it/s]

In [20]:
navis.plot3d(nl)

In [21]:
output_path2 = FISBe_DIR /"completely/train"/ "swc_output 2"
print(output_path2)

/Users/vuhepola/GitHub/Repos/neuroinfo_fruitfly/data/FISBe/completely/train/swc_output 2


In [22]:
nl2 = navis.read_swc(output_path2 / f"{file}/*.swc")

Importing:   0%|          | 0/2 [00:00<?, ?it/s]

In [24]:
navis.plot3d(nl2)

In [27]:
sample = navis.make_dotprops(nl2)

navis.plot3d(sample)

Dotprops:   0%|          | 0/2 [00:00<?, ?it/s]

  File "<frozen runpy>", line 198, in _run_module_as_main
  File "<frozen runpy>", line 88, in _run_code
  File "/opt/miniconda3/envs/neurofly/lib/python3.12/site-packages/ipykernel_launcher.py", line 18, in <module>
    app.launch_new_instance()
  File "/opt/miniconda3/envs/neurofly/lib/python3.12/site-packages/traitlets/config/application.py", line 1082, in launch_instance
    app.start()
  File "/opt/miniconda3/envs/neurofly/lib/python3.12/site-packages/ipykernel/kernelapp.py", line 739, in start
    self.io_loop.start()
  File "/opt/miniconda3/envs/neurofly/lib/python3.12/site-packages/tornado/platform/asyncio.py", line 211, in start
    self.asyncio_loop.run_forever()
  File "/opt/miniconda3/envs/neurofly/lib/python3.12/asyncio/base_events.py", line 645, in run_forever
    self._run_once()
  File "/opt/miniconda3/envs/neurofly/lib/python3.12/asyncio/base_events.py", line 1999, in _run_once
    handle._run()
  File "/opt/miniconda3/envs/neurofly/lib/python3.12/asyncio/events.py", l

In [8]:
# Convert the full node DataFrame to a numpy array
coords_array = nl.vertices
coords_array.shape

(2,)